# 01 — Download BCSS, and prove the release is the right one

**Gates this notebook closes: G0 and G1.**

Two jobs, and the first one is not about BCSS at all.

**G0 — the smoke test.** Everything in this plan rests on one thing: that a notebook can
import the demo backend and get *the same haematoxylin channel the API would*. If a training
script quietly computes its own H channel while the runtime uses the calibrated one, the two
drift apart and the IHC slides score nothing like the H&E — and it looks like a model problem
for a week before anyone finds the plumbing.

**G1 — the release check.** There is a widely-used "5-class" version of BCSS that **merges
DCIS into tumour**. That is the one merge this project cannot accept: separating in-situ from
invasive carcinoma is the entire point of the region model. This notebook refuses to continue
unless code `20 dcis` is present and distinct from code `1 tumor`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import backend_path            # noqa: F401 - puts the demo backend on sys.path
backend_path.ensure_dirs()
print("backend :", backend_path.BACKEND_ROOT)
print("tiles   :", backend_path.TILES_DIR)
print("models  :", backend_path.MODELS_DIR)

backend : C:\Users\Coditas\Desktop\Healthcare_Projects\Cancer_Scoring_System\Breast_Cancer_IHC_Tissue_Scoring_Demo\backend
tiles   : C:\Users\Coditas\Desktop\Healthcare_Projects\Cancer_Scoring_System\bcss_hchannel_resnet18\data\tiles
models  : C:\Users\Coditas\Desktop\Healthcare_Projects\Cancer_Scoring_System\Breast_Cancer_IHC_Tissue_Scoring_Demo\models\tissue_type


## G0 — the H channel, two ways

`hchannel.haematoxylin_od` must agree with un-mixing by hand through step 6's own fixed basis.
This is not a tautology: it is the assertion that nothing sits between the pipeline's optical
density and the pipeline's stain matrix.

In [2]:
import numpy as np

import hchannel
from app.common.imaging import optical_density
from app.common.stains import REFERENCE_BY_NAME, RUIFROK_INVERSE, unmix

# A synthetic tile of pure haematoxylin at a known concentration, built from
# Beer-Lambert: I = I0 * 10^(-c*v). The expected answer is therefore known exactly.
vector = np.asarray(REFERENCE_BY_NAME["haematoxylin"])
rgb = np.tile((255.0 * 10.0 ** (-0.8 * vector)).astype(np.uint8), (64, 64, 1))
white = (255.0, 255.0, 255.0)

ours = hchannel.haematoxylin_od(rgb, white)
od = optical_density(rgb, white)
by_hand = unmix(od.reshape(-1, 3), RUIFROK_INVERSE)[:, 0].reshape(od.shape[:2])

print(f"ours      mean H = {ours.mean():.4f}")
print(f"by hand   mean H = {by_hand.mean():.4f}")
print(f"expected         = 0.8000  (8-bit quantisation is the only error source)")
assert np.allclose(ours, by_hand, atol=1e-6), "the notebook and the pipeline disagree"
print("\nG0a OK - one definition of the haematoxylin channel")

ours      mean H = 0.8016
by hand   mean H = 0.8016
expected         = 0.8000  (8-bit quantisation is the only error source)

G0a OK - one definition of the haematoxylin channel


### The same check on a real slide

Synthetic pixels prove the arithmetic. A real tile proves the *plumbing*: the pyramid, the
level rule (resolution chosen in microns, never by level index), and the BOX resampling that
has to happen in intensity space before the logarithm.

In [4]:
from app.ingestion.slide_reader import open_slide
from app.pipeline.step05_optical_density.tiles import read_tile

slides = sorted(backend_path.SLIDES_DIR.glob("*.svs"))
print(f"{len(slides)} slide(s) available")

if slides:
    reader = open_slide(slides[0])
    print(f"  {slides[0].name}: {reader.dimensions} at {reader.mpp} um/px, "
          f"{reader.level_count} levels")

    # Middle of the canvas, which on an Aperio frame is where the section usually is.
    width, height = reader.dimensions
    tile = read_tile(reader, x=width // 2, y=height // 2, target_mpp=0.5,
                     size=224, base_mpp=reader.mpp)
    print(f"  tile: {tile.size}px at {tile.mpp:.4f} um/px from level {tile.level}"
          f"{' (resampled)' if tile.resampled else ''}")

    h = hchannel.haematoxylin_od(tile.rgb, hchannel.white_point(tile.rgb))
    print(f"  H channel: mean {h.mean():.3f}, p99.5 {np.percentile(h, 99.5):.3f} OD")
    print("\nG0b OK - a real tile goes through the real pipeline")
else:
    print("  no slides on disk - G0b skipped, which is fine for training on BCSS alone")

16 slide(s) available


ModuleNotFoundError: No module named 'tiffslide'

## Download BCSS

~5–10 GB, CC0, 151 regions from 151 TCGA-BRCA slides at 0.25 µm/px.

Route 1 is the Google Drive folder the BCSS repository links to. If Drive rate-limits, see
`src/download_bcss.py` for routes 2 (the authors' girder script) and 3 (a Kaggle mirror) —
and note that **mirrors of the merged 5-class version exist**, which is what the next cell
is for.

> **Disk.** Check you have room before starting. The download is ~10 GB and the tile export
> needs a few hundred MB more; BCSS itself can be deleted once gate G2 passes.

In [ ]:
import shutil

free_gb = shutil.disk_usage(backend_path.DATA_DIR).free / 1e9
print(f"free on this drive: {free_gb:.1f} GB")
if free_gb < 15:
    print("  ! under 15 GB free. BCSS is ~10 GB and the export needs headroom.")
    print("  ! free some space, or point BCSS_DIR at another drive, before continuing.")

In [ ]:
import download_bcss

# The label codes first: tiny, and the file every mapping in bcss.py is checked against.
codes_path = download_bcss.fetch_gtruth_codes(backend_path.BCSS_DIR / "meta" / "gtruth_codes.tsv")
codes = download_bcss.check_codes_file(codes_path)
print(f"gtruth_codes.tsv: {len(codes)} codes, and they match bcss.GT_CODES")
print(f"  tumor = {codes['tumor']}, dcis = {codes['dcis']}  <- these must differ")

In [ ]:
# The regions themselves: ~4 GB, resumable per region, ~1 hour.
#
# `--route girder` in the script, and `download_girder` here, because the Google Drive
# folder the BCSS README links to RATE-LIMITS: it served 42 of 302 files and then
# answered everything else with "Cannot retrieve the public link ... or have had many
# accesses". This route crops each ROI server-side out of its whole slide and takes the
# mask from figshare, so image and mask arrive in the SAME coordinate space - level-0
# pixels of the source slide - and align exactly with no resampling on either side.
#
# It also records each slide's own resolution into data/bcss/slide_mpp.json, which the
# exporter requires. See the next cell for why that is not optional.
root = download_bcss.download_girder(backend_path.BCSS_DIR)
print("images and masks in", root)

## G1 — is this the raw 22-class release, and is it complete?

`verify()` raises if code 20 is absent (the merged release) or if an image has no mask (a
partial download). Both failures produce a model that trains fine and means something
different from what its report claims, which is why they are refusals rather than warnings.

In [ ]:
summary = download_bcss.verify(root, sample_masks=20)

print(f"regions      : {summary['regions']} (expected 151)")
print(f"slides       : {summary['slides']}")
print(f"institutions : {len(summary['institutions'])}")
print(f"train / test : {summary['train_regions']} / {summary['held_out_regions']}"
      f"  (held out: OL LL E2 EW GM S3)")
print(f"on disk      : {summary['bytes_on_disk'] / 1e9:.2f} GB")
print()
print(f"dcis pixels in the sampled masks : {summary['dcis_pixels_sampled']:,}")
print(f"tumor pixels in the sampled masks: {summary['tumor_pixels_sampled']:,}")

print()
print("code census over the sample:")
for name, count in sorted(summary["code_census"].items(), key=lambda kv: -kv[1]):
    print(f"  {name:<26} {count:>12,}")

manifest = download_bcss.write_manifest(backend_path.BCSS_DIR, summary)
print()
print(f"wrote {manifest.name}")
print("G1 OK - raw release, dcis separate from tumor")

# --- the resolution trap ----------------------------------------------------------
#
# EVERY FILENAME IN THIS DATASET ENDS "MPP-0.2500". THE SLIDES ARE NOT 0.25 um/px.
#
# The Drive folder is even called 0_Public-data-Amgad2019_0.25MPP, so every tutorial
# assumes 0.25 - and so did the first draft of this plan. Measured from the slides own
# tile metadata they are 0.2505, 0.2521, and so on: TCGA scans vary.
#
# It matters because a 224 px tile is meant to cover 112 um - about nine cells across -
# and that is the entire argument for the tile size. Nine cells is a claim about
# microns, so the microns have to be measured rather than read off a filename.
measured = download_bcss.resolutions(backend_path.BCSS_DIR)
spread = sorted({round(v, 4) for v in measured.values()})
print()
print(f"measured resolutions across {len(measured)} slides: {spread}")
print("the filenames all claim 0.2500, and they are wrong")

## Eyeball three regions

The last check a human can do that a test cannot: does the mask line up with the image, and
do the three classes look like what they claim to be? Class `0` fat and stroma are
unmistakable to a non-expert — if class `2` tiles look like empty space, the mask is
misaligned and everything downstream is being trained on noise.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

import bcss

Image.MAX_IMAGE_PIXELS = 500_000_000
regions = bcss.find_regions(backend_path.BCSS_DIR)
picks = [regions[0], regions[len(regions) // 2], regions[-1]]

# Three classes plus ignore, in the colours the demo overlay uses.
palette = np.array([[0.86, 0.90, 0.96], [0.99, 0.85, 0.60], [0.85, 0.25, 0.30], [1, 1, 1]])

fig, axes = plt.subplots(len(picks), 3, figsize=(13, 4.2 * len(picks)))
for row, region in enumerate(picks):
    rgb = np.asarray(Image.open(region.image).convert("RGB"))
    remapped = bcss.remap(np.asarray(Image.open(region.mask)))
    h = hchannel.haematoxylin_od(rgb, hchannel.white_point(rgb))

    coloured = palette[np.where(remapped == bcss.IGNORE, 3, remapped)]

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"{region.roi_id[:28]}\n{region.institution}"
                           f"{' (held out)' if region.is_test else ''}", fontsize=9)
    axes[row, 1].imshow(np.clip(h, 0, 1.5) / 1.5, cmap="bone_r")
    axes[row, 1].set_title("haematoxylin channel - what the model sees", fontsize=9)
    axes[row, 2].imshow(coloured)
    shares = {bcss.CLASS_NAMES[c]: float((remapped == c).mean()) for c in (0, 1, 2)}
    axes[row, 2].set_title("blue non-epi | amber in-situ | red invasive\n"
                           + "  ".join(f"{k.split('_')[0]} {v:.0%}" for k, v in shares.items()),
                           fontsize=9)
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(backend_path.REPORTS_DIR / "01_bcss_regions.png", dpi=110, bbox_inches="tight")
plt.show()

---

**Both gates closed?** Then notebook 02 is worth writing.

- [ ] G0 — the notebook's H channel is the pipeline's H channel
- [ ] G1 — `dcis` is code 20, present, and not merged into `tumor`
- [ ] 151 regions, every image paired with a mask
- [ ] the three panels above look like tissue, and the mask lines up with the image